# talking-avatar — Google Colab setup

Confirmed working end-to-end (voice cloning + Ditto animation) on a
Colab T4, as of 2026-08-08. Run cells top to bottom, in order.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

## 1. Install conda (condacolab) — restarts the runtime automatically

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()
# The runtime restarts automatically after this cell. Just continue
# running the cells below once it comes back -- no need to re-run this one.

## 2. Mount Google Drive

Keeps downloaded model weights across session resets, so you're not re-downloading multiple GB every time Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/talking_avatar_models', exist_ok=True)

## 3. Clone the repo and install main-environment dependencies

In [ ]:
!git clone https://github.com/Chirag514/talking-avatar.git
%cd talking-avatar

In [ ]:
!pip install -q --upgrade "transformers>=5.3.0" soundfile
!pip install -q omnivoice --no-deps

# omnivoice's --no-deps skips its real dependency list (undocumented
# upstream). CONFIRMED complete list for omnivoice 0.2.1, sourced from
# pip's own dependency-conflict warning:
!pip install -q torchaudio  # unpinned -- matches Colab's newer torch via torchaudio's stable ABI (2.11+)
!pip install -q accelerate gradio librosa pydub tensorboardx webdataset

!pip install -q openai-whisper openai groq moviepy pillow opencv-python playwright
!playwright install --with-deps chromium
!apt-get install -qq -y fonts-noto-core imagemagick

## 4. Real-ESRGAN (restoration stage)

Cloned onto Drive so it persists across sessions.

In [ ]:
![ ! -d "/content/drive/MyDrive/talking_avatar_models/Real-ESRGAN" ] && \
  git clone https://github.com/xinntao/Real-ESRGAN.git /content/drive/MyDrive/talking_avatar_models/Real-ESRGAN

%cd /content/drive/MyDrive/talking_avatar_models/Real-ESRGAN
!pip install -q -r requirements.txt && python setup.py develop
!mkdir -p weights && wget -q -O weights/RealESRGAN_x4plus.pth \
    https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
%cd /content/talking-avatar

## 5. Ditto's conda environment

Uses `environment_colab.yaml`, **not** `environment.yaml` — the full
`environment.yaml` pins `tensorrt==8.6.1`, which doesn't exist for
Colab's Python/CUDA combination and fails the whole env creation.
`environment_colab.yaml` is a minimal, CONFIRMED-working subset.

In [ ]:
!conda env create -f pipeline/ditto_talkinghead/environment_colab.yaml
!conda env list  # confirm 'ditto' shows up with a real path before continuing

## 6. Fix torch's CUDA build

CONFIRMED necessary: conda's pytorch install reports
`torch.cuda.is_available() == False` on Colab — a driver/build
mismatch, not a code issue. Force-reinstall against Colab's actual
CUDA build.

In [ ]:
!conda run -n ditto pip uninstall -y torch torchvision torchaudio
!conda run -n ditto pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

# should print CUDA available: True
!conda run -n ditto python -c "import torch; print('CUDA available:', torch.cuda.is_available())" 

## 7. Remaining Ditto dependencies

What `environment.yaml`'s pip section would normally install, applied manually against the working env.

In [ ]:
!conda run -n ditto pip install -q audioread==3.0.1 cffi==1.17.1 cuda-python==12.6.2.post1 \
    cython==3.0.11 decorator==5.1.1 filetype==1.2.0 imageio==2.36.1 \
    imageio-ffmpeg==0.5.1 joblib==1.4.2 lazy-loader==0.4 \
    librosa==0.10.2.post1 llvmlite==0.43.0 msgpack==1.1.0 numba==0.60.0 \
    nvidia-cublas-cu12==12.6.4.1 nvidia-cuda-runtime-cu12==12.6.77 \
    nvidia-cudnn-cu12==9.6.0.74 opencv-python-headless==4.10.0.84 \
    packaging==24.2 platformdirs==4.3.6 pooch==1.8.2 pycparser==2.22 \
    scikit-image==0.25.0 scikit-learn==1.6.0 scipy==1.15.0 \
    soundfile==0.13.0 soxr==0.5.0.post1 threadpoolctl==3.5.0 \
    tifffile==2024.12.12 tqdm==4.67.1 polygraphy colored
!conda run -n ditto pip install -q einops mediapipe
# pyximport comes bundled with cython above -- no separate install needed

## 8. Swap onnxruntime for the GPU build

CONFIRMED necessary: the plain `onnxruntime` pip package is CPU-only.

In [ ]:
!conda run -n ditto pip uninstall -y onnxruntime
!conda run -n ditto pip install onnxruntime-gpu
!conda run -n ditto python -c "import onnxruntime as ort; print(ort.get_available_providers())" 

## 9. Ditto checkpoints

In [ ]:
!git lfs install
![ ! -d "pipeline/ditto_talkinghead/checkpoints" ] && \
  git clone https://huggingface.co/digital-avatar/ditto-talkinghead pipeline/ditto_talkinghead/checkpoints

## 10. API keys

`OPENAI_API_KEY` is only needed if you're running Stage 1 (safety
gate). For dev/testing, `run_pipeline.py --skip_safety_gate` skips it
entirely. `GROQ_API_KEY` is only needed for `--overlays`.

**Don't hardcode real keys in a notebook cell** — use Colab's Secrets
manager (key icon in the left sidebar) and pull them in with
`from google.colab import userdata; userdata.get('OPENAI_API_KEY')`,
or `getpass` if you'd rather paste it fresh each session.

In [ ]:
import os
from getpass import getpass

# Leave blank / skip if using --skip_safety_gate and not testing --overlays
os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (blank to skip): ") or ""
os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY (blank to skip): ") or ""

os.environ["MPLBACKEND"] = "Agg"  # headless matplotlib backend

## 11. Verify the environment before testing

Confirms every package that's bitten us before is actually present.

In [ ]:
!conda run -n ditto python -c "import torch, filetype, pyximport, onnxruntime, mediapipe, einops; \
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available()); \
print('All ditto-env imports OK')" 

## 12. Test stages individually

Upload your own reference image / reference voice clip / script to
`/content` first (or point at paths already on Drive), then run
whichever of these you need. Test before attempting the full pipeline
— failures are much cheaper to diagnose per-stage.

In [ ]:
SCRIPT = "/content/script.txt"
REF_IMAGE = "/content/reference_image.png"
REF_VOICE = "/content/reference_voice.wav"

!bash test_stages.sh voice {SCRIPT} {REF_VOICE} --speed 1.4

In [ ]:
# Run inside the ditto env
!conda run -n ditto bash test_stages.sh ditto {REF_IMAGE} test_output/02_voice_speed1.4.wav

In [ ]:
!bash test_stages.sh restore test_output/ditto_test.mp4

In [ ]:
!bash test_stages.sh export test_output/restored.mp4 test_output/02_voice_speed1.4.wav test_output/final.mp4

## 13. Full pipeline (once individual stages pass)

Write output straight to a Drive path if you want it to survive a
runtime disconnect.

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/talking_avatar_models/output.mp4"

!conda run -n ditto python pipeline/run_pipeline.py \
    {SCRIPT} {REF_IMAGE} {REF_VOICE} {OUTPUT_PATH} \
    --submitting_user_id colab_test --speed 1.4 --skip_safety_gate

---
### Troubleshooting quick reference

- **`DirectoryNotACondaEnvironmentError`** — env creation (step 5) failed
  early and left a broken stub. `!rm -rf /usr/local/envs/ditto`, then
  retry step 5 and actually read the output before moving on.
- **`ModuleNotFoundError: No module named 'pyximport'`** — `cython`
  didn't install in step 7 (pyximport ships inside the cython package,
  not separately).
- **`torch.cuda.is_available()` is `False`** — you skipped step 6, or
  it needs to be re-run after step 5.
- **Commands running against the wrong environment** — the `ditto`
  subcommand of `test_stages.sh` and anything touching Ditto must be
  prefixed with `conda run -n ditto`, or it silently runs against your
  main environment instead and fails on missing packages one at a time.
